# 5. Evaluation & Error Analysis
Simple evaluation of all models from notebook 4: metrics table, confusion matrices, feature importance, and key plots.

In [13]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
)

ROOT = Path(globals()["__vsc_ipynb_file__"]).parent.parent
TRAIN_PATH = ROOT / "data" / "processed" / "train.csv"
TEST_PATH = ROOT / "data" / "processed" / "test.csv"
BEST_MODEL_CONFIG_PATH = ROOT / "results" / "best_model_config.json"

# Load data (same split as notebook 4)
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

y_train = train_df["TARGET_CLASS"].astype(int)
y_test = test_df["TARGET_CLASS"].astype(int)
X_train = train_df.drop(columns=["TARGET_CLASS"])
X_test = test_df.drop(columns=["TARGET_CLASS"])

target_le = LabelEncoder()
y_train = pd.Series(target_le.fit_transform(y_train), index=y_train.index)
y_test = pd.Series(target_le.transform(y_test), index=y_test.index)

labels = sorted(y_test.unique())
label_names = [str(target_le.inverse_transform([l])[0]) for l in labels]

# Load best model config
best_cfg = json.loads(BEST_MODEL_CONFIG_PATH.read_text(encoding="utf-8"))
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {label_names}")
print(f"Best model from notebook 4: {best_cfg['model']} ({best_cfg['variant']})")

Train: (1204, 18), Test: (302, 18)
Classes: ['1', '2', '3', '4']
Best model from notebook 4: ExtremeBoosting (tuned)


In [ ]:
# Define all models (baseline + tuned best from notebook 4)
models = {
    ("LogisticRegression", "baseline"): LogisticRegression(max_iter=5000, random_state=42),
    ("DecisionTree", "baseline"): DecisionTreeClassifier(random_state=42),
    ("RandomForest", "baseline"): RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
    ("XGBoost", "baseline"): XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss",
        n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42),
    ("ExtremeBoosting", "baseline"): XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss",
        n_estimators=500, learning_rate=0.03, max_depth=8,
        min_child_weight=3, reg_lambda=1.5, random_state=42),
}

# Add tuned best model from notebook 4 config
best_name = best_cfg["model"]
best_params = best_cfg.get("params", {})
model_classes = {
    "LogisticRegression": lambda p: LogisticRegression(**p, max_iter=7000, random_state=42),
    "DecisionTree": lambda p: DecisionTreeClassifier(**p, random_state=42),
    "RandomForest": lambda p: RandomForestClassifier(**p, random_state=42, n_jobs=-1),
    "XGBoost": lambda p: XGBClassifier(**p, objective="multi:softprob", eval_metric="mlogloss", random_state=42),
    "ExtremeBoosting": lambda p: XGBClassifier(**p, objective="multi:softprob", eval_metric="mlogloss", random_state=42),
}
if best_name in model_classes:
    models[(best_name, "tuned")] = model_classes[best_name](best_params)

# Train all models and collect metrics
rows = []
trained_models = {}
for (name, variant), model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    trained_models[(name, variant)] = (model, y_pred)
    rows.append({
        "model": name, "variant": variant,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    })

results_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
results_df.index += 1
results_df.index.name = "rank"
print("Model Evaluation Ranking")
results_df.round(4)

TypeError: RandomForestClassifier.__init__() got an unexpected keyword argument 'subsample'

In [ ]:
# Classification report for each model
for (name, variant), (model, y_pred) in trained_models.items():
    print(f"\n{'='*50}")
    print(f"{name} ({variant})")
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

In [ ]:
# Confusion matrices for all models
import math

n_models = len(trained_models)
n_cols = 3
n_rows = math.ceil(n_models / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 4.5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, ((name, variant), (model, y_pred)) in enumerate(trained_models.items()):
    ax = axes[idx]
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.2f})",
                    ha="center", va="center", fontsize=8)

    ax.set_title(f"{name} ({variant})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(label_names)))
    ax.set_xticklabels(label_names, fontsize=8)
    ax.set_yticks(range(len(label_names)))
    ax.set_yticklabels(label_names, fontsize=8)

for idx in range(n_models, len(axes)):
    axes[idx].axis("off")

plt.suptitle("Confusion Matrices (count + row-normalized)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for all models
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, ((name, variant), (model, _)) in enumerate(trained_models.items()):
    ax = axes[idx]
    if hasattr(model, "feature_importances_"):
        imp = model.feature_importances_
        method = "feature_importances_"
    elif hasattr(model, "coef_"):
        imp = np.mean(np.abs(model.coef_), axis=0)
        method = "|coef|"
    else:
        imp = np.zeros(X_train.shape[1])
        method = "N/A"

    top_idx = np.argsort(imp)[-15:]
    ax.barh(np.array(X_train.columns)[top_idx], imp[top_idx], color="#2a6f97")
    ax.set_title(f"{name} ({variant})\n[{method}]", fontsize=10)
    ax.set_xlabel("Importance")

for idx in range(n_models, len(axes)):
    axes[idx].axis("off")

plt.suptitle("Top 15 Feature Importances per Model", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison bar chart
results_plot = results_df.reset_index(drop=True)
results_plot["label"] = results_plot["model"] + " (" + results_plot["variant"] + ")"

fig, ax = plt.subplots(figsize=(12, 6))
metrics = ["f1_macro", "accuracy", "precision_macro", "recall_macro"]
x = np.arange(len(results_plot))
width = 0.2

for i, m in enumerate(metrics):
    ax.bar(x + i * width, results_plot[m], width, label=m)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_plot["label"], rotation=30, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Model Comparison")
ax.legend()
plt.tight_layout()
plt.show()